In [12]:
import pandas as pd
import matplotlib.pyplot as plt

from banco import get_engine, executar

engine = get_engine()
pd.set_option("display.max_colwidth", 80)
pd.options.display.float_format = "{:,.2f}".format

def consulta(sql):
    return pd.read_sql(sql, engine)

def grafico_barras(df, rotulo, valor, titulo, eixo_x):
    d = df.sort_values(valor).copy()
    d[rotulo] = d[rotulo].astype(str).str.slice(0,45)
    fig, ax = plt.subplots(figsize=(10, 0.6 * len(d) + 2))
    ax.barh(d[rotulo], d[valor].astype(float))
    ax.set_title(titulo)
    ax.set_xlabel(eixo_x)
    ax.ticklabel_format(axis="x", style="plain")
    plt.tight_layout()
    plt.show()

Configurações do banco: {'user': 'postgres', 'password': '123456', 'host': 'localhost', 'port': '5432', 'database': 'viagens_servico'}


In [ ]:
#Part 1 - Perguntas respondidas na Silver
# 1 - Quais os 5 órgãos com maior custo total?
SQL_Q1 = """
SELECT 
    nome_orgao_superior,
    count(*) as qtd_viagens,
    sum(valor_total) as custo_total
FROM 
silver_viagem
GROUP BY nome_orgao_superior
ORDER BY custo_total DESC
LIMIT 5
"""
q1 = consulta(SQL_Q1)
display(q1)
grafico_barras(q1, "nome_orgao_superior", "custo_total",
               "Top 5 órgãos por custo total de viagens", "Custo Total (R$)")


In [ ]:
# 2 - Quais os 3 destinos com maior custo médio por viagem?
SQL_Q2 = """
SELECT 
    destinos,
    count(*) as qtd_viagens,
    avg(valor_total) as custo_medio
FROM
silver_viagem
where
destinos is not null
group by destinos
order by custo_medio desc
limit 3
"""
q2 = consulta(SQL_Q2)
display(q2)
grafico_barras(q2, "destinos", "custo_medio",
               "Top 3 destinos por custo médio por viagem", "Custo médio (R$)")

In [ ]:
SQL_Q2B = """

SELECT 
    destinos,
    count(*) as qtd_viagens,
    avg(valor_total) as custo_medio
FROM
silver_viagem
where
destinos is not null
group by destinos
having count(*) >= 30
order by custo_medio desc
limit 3
"""
display(consulta(SQL_Q2B))

In [ ]:
# 3 - Qual a viagem de maior duração e qual o seu custo total?
SQL_Q3 = """

SELECT 
    id_viagem, 
    nome_orgao_superior, 
    data_inicio, 
    data_fim,
    duracao_dias, 
    valor_total
FROM silver_viagem
where
duracao_dias is not null
order by duracao_dias desc, valor_total desc
limit 10
"""

q3 = consulta(SQL_Q3)
display(q3)
top = q3.iloc[0]
print(f"Resposta: viagem {top['id_viagem']} - {top['duracao_dias']} dias, "
      f"custo total R$ {top['valor_total']:,.2f}")
grafico_barras(q3, "id_viagem", "duracao_dias",
               "10 viagens de maior duração", "Duração (dias)")


In [ ]:
#Parte 2 - Camada Gold
#Duas tabelas agregadas com JOIN + GROUP BY
GOLD_PAGAMENTOS = """
SELECT
    p.tipo_pagamento,
    p.nome_orgao_pagador,
    count(*) as qtd_pagamentos,
    sum(p.valor) as valor_total,
    round(avg(p.valor), 2) as valor_medio
FROM
silver_pagamento p
JOIN silver_viagem v ON v.id_viagem = p.id_viagem
GROUP BY p.tipo_pagamento, p.nome_orgao_pagador

"""
GOLD_TRECHOS = """
SELECT 
    t.meio_transporte,
    t.destino_uf,
    count(*) as qtd_trechos
FROM
silver_trecho t
JOIN silver_viagem v ON v.id_viagem = t.id_viagem
GROUP BY t.meio_transporte, t.destino_uf
"""

for nome, sql in {"gold_pagamentos": GOLD_PAGAMENTOS,
                  "gold_trechos":GOLD_TRECHOS}.items():
    executar(f"DROP TABLE IF EXISTS {nome}")
    executar(f"CREATE TABLE {nome} AS {sql}")
    executar(f"CREATE OR REPLACE VIEW vw_{nome} AS {sql}")

display(consulta("SELECT * FROM gold_pagamentos LIMIT 5"))
display(consulta("SELECT * FROM gold_trechos limit 5"))



In [ ]:
# 4 - Qual o tipo de pagamento com maior valor médio?
SQL_Q4 = """
SELECT 
    tipo_pagamento,
    sum(qtd_pagamentos) as qtd_pagamentos,
    sum(valor_total) / sum(qtd_pagamentos) as valor_medio
FROM
gold_pagamentos
GROUP BY tipo_pagamento
order by valor_medio desc

"""
q4 = consulta(SQL_Q4)
display(q4)
print(f"Resposta: {q4.iloc[0]['tipo_pagamento']}")
grafico_barras(q4, "tipo_pagamento", "valor_medio",
               "Valor médio por tipo de pagamento", "Valor médio (R$)")


In [ ]:
# 5 - Qual o meio de transporte mais usado nos trechos?
SQL_Q5 = """
SELECT
    meio_transporte,
    sum(qtd_trechos) as qtd_trechos
FROM
gold_trechos
GROUP BY meio_transporte
order by qtd_trechos desc

"""
q5 = consulta(SQL_Q5)
display(q5)
print(f"Resposta: {q5.iloc[0]['meio_transporte']}")
grafico_barras(q5, "meio_transporte", "qtd_trechos",
               "Trechos por meio de transporte", "Quantidade de trechos")

In [ ]:
# 6 - Qual UF de destino aparece em mais trechos?
SQL_Q6 = """
SELECT 
    destino_uf,
    sum(qtd_trechos) as qtd_trechos
    
FROM
gold_trechos
GROUP BY destino_uf
ORDER BY qtd_trechos desc
limit 10

"""
q6 = consulta(SQL_Q6)
display(q6)
print(f"Resposa: {q6.iloc[0]['destino_uf']}")
grafico_barras(q6, "destino_uf", "qtd_trechos",
               "Top 10 UFs de destino por número de trechos",
               "Quantidade de trechos")


In [24]:
# 7 - Qual órgão pagou mais no total?
SQL_Q7 = """
select 
    nome_orgao_pagador,
    sum(valor_total) as valor_total
FROM
gold_pagamentos
group by nome_orgao_pagador
order by valor_total desc

"""
q7 = consulta(SQL_Q7)
display(q7)
print(f"resposta: {q7.iloc[0]['nome_orgao_pagador']} - "
      f"R$ {q7.iloc[0]['valor_total']:,.2f}")



,nome_orgao_pagador,valor_total
0,Fundo Nacional de Segurança Pública,"278,481,047.89"
1,Sigiloso,"200,484,801.68"
2,Comando da Aeronáutica,"81,769,144.77"
3,Instituto Nacional do Seguro Social,"37,427,601.45"
4,Comando do Exército,"36,872,643.95"
...,...,...
211,"Instituto Federal de Educação, Ciência e Tecnologia de Brasília","32,877.29"
212,Fundação Osório,"8,737.50"
213,Fundo Nacional sobre Mudança no Clima,"4,821.45"
214,Fundo Penitenciário Nacional,"1,918.59"


resposta: Fundo Nacional de Segurança Pública - R$ 278,481,047.89
